# Notebook 02 — Data Cleaning and Preprocessing

**Project:** Air Guitar CS249R (Software-Only ML Architecture)

This notebook cleans raw hand landmark observations, computes invariant features, and packages sequences.

### Preprocessing Pipeline:
1. **Quality Check:** Detect missing landmarks, NaN/infinite values, and invalid coordinates.
2. **Geometric Normalization:**
   - Translate wrist $L_0$ to origin $(0, 0, 0)$ (Translation Invariance).
   - Scale by distance $\|L_9 - L_0\|$ (Palm Scale Invariance).
3. **Feature Engineering:**
   - 20 anatomical Euclidean distances (fingertips to wrist, knuckles, adjacent fingers).
   - 15 joint flexion angles in radians (capturing curl of all 5 fingers).
   - Total = **98 engineered features**.
4. **Temporal Sequencing (Strumming):** Sliding 15-frame sequence window preserving session boundaries.
5. **Non-Leakage Splitting:** Partition by `session_id` to prevent cross-contamination.
6. **Artifact Persistence:** Save to `dataset/processed/chord_features.csv`, `strumming_sequences.npz`, `metadata.json`.

In [11]:
# 1. Imports and Paths
import sys
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
import joblib

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import (
    RAW_CHORD_CSV, RAW_STRUM_CSV, PROCESSED_DATA_DIR,
    PROCESSED_CHORD_CSV, PROCESSED_STRUM_NPZ, PROCESSED_METADATA_JSON,
    SMOKE_TEST_DATA_DIR, CHORD_MODEL_DIR, STRUM_MODEL_DIR,
    SUPPORTED_CHORDS, STRUM_CLASSES, SEQUENCE_LENGTH
)
from src.feature_extraction import (
    extract_chord_features, extract_strum_frame_features, get_chord_feature_names
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Preprocessing directories initialized.')

Preprocessing directories initialized.


## 2. Load Raw Chord Data or Synthetic Smoke Fixture

If real webcam samples have been collected in `dataset/raw/`, they are processed; otherwise, the verified smoke fixture is loaded.

In [12]:
# 2. Load chord dataset
use_smoke_fixture = not (RAW_CHORD_CSV.exists() and RAW_CHORD_CSV.stat().st_size > 500)
if use_smoke_fixture:
    print('[Notice] No real webcam chord samples found in dataset/raw/. Loading synthetic smoke fixture.')
    smoke_chord_path = SMOKE_TEST_DATA_DIR / 'smoke_chord_features.csv'
    df_chords = pd.read_csv(smoke_chord_path)
    print(f'Loaded {len(df_chords)} smoke chord rows.')
else:
    print(f'Loading real collected raw chord samples from {RAW_CHORD_CSV}...')
    raw_df = pd.read_csv(RAW_CHORD_CSV)
    coord_cols = [f'{axis}_{i}' for i in range(21) for axis in ('x', 'y', 'z')]
    
    processed_rows = []
    feat_names = get_chord_feature_names()
    for _, row in raw_df.iterrows():
        coords = row[coord_cols].values.astype(np.float32).reshape(21, 3)
        features = extract_chord_features(coords)
        row_dict = {'session_id': row['session_id'], 'label': row['label']}
        for fn, fv in zip(feat_names, features):
            row_dict[fn] = fv
        processed_rows.append(row_dict)
    df_chords = pd.DataFrame(processed_rows)
    print(f'Engineered features for {len(df_chords)} real chord samples.')

Loading real collected raw chord samples from C:\Users\piyus\Desktop\ds cp airguitar\dataset\raw\chord_landmarks.csv...
Engineered features for 871 real chord samples.


## 3. Label Encoding and Feature Schema

Encode chord labels into integers and persist the `LabelEncoder`.

In [13]:
# 3. Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(SUPPORTED_CHORDS)
df_chords['label_encoded'] = label_encoder.transform(df_chords['label'])

# Save encoder and schema
CHORD_MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(label_encoder, CHORD_MODEL_DIR / 'label_encoder.joblib')

feature_names = get_chord_feature_names()
schema = {'features': feature_names, 'num_features': len(feature_names)}
with open(CHORD_MODEL_DIR / 'feature_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

# Save processed chord dataset
df_chords.to_csv(PROCESSED_CHORD_CSV, index=False)
print(f'Saved processed chord features to {PROCESSED_CHORD_CSV}')
print('Class distribution:')
print(df_chords['label'].value_counts())

Saved processed chord features to C:\Users\piyus\Desktop\ds cp airguitar\dataset\processed\chord_features.csv
Class distribution:
label
c     91
cm    84
a     72
d     70
bm    69
b     66
gm    65
em    63
g     63
e     61
dm    60
f     56
am    51
Name: count, dtype: int64


## 4. Strumming Temporal Sequences

Process strumming sequences of 15 frames.

In [14]:
# 4. Process Strumming Sequences
use_smoke_strum = not (RAW_STRUM_CSV.exists() and RAW_STRUM_CSV.stat().st_size > 500)
if use_smoke_strum:
    print('[Notice] Loading synthetic smoke strum sequences...')
    smoke_strum_data = np.load(SMOKE_TEST_DATA_DIR / 'smoke_strumming_sequences.npz')
    X_strum = smoke_strum_data['X']
    y_strum = smoke_strum_data['y']
    sess_strum = smoke_strum_data['session_ids']
else:
    print(f'Processing real raw strum frames from {RAW_STRUM_CSV}...')
    raw_s = pd.read_csv(RAW_STRUM_CSV)
    # Group by (session_id, sequence_id)
    X_list, y_list, sess_list = [], [], []
    coord_cols = [f'{axis}_{i}' for i in range(21) for axis in ('x', 'y', 'z')]
    
    for (s_id, seq_id), group in raw_s.groupby(['session_id', 'sequence_id']):
        if len(group) >= SEQUENCE_LENGTH:
            group_sorted = group.sort_values('frame_idx').iloc[:SEQUENCE_LENGTH]
            frames = []
            prev_lm = None
            for _, row in group_sorted.iterrows():
                curr_lm = row[coord_cols].values.astype(np.float32).reshape(21, 3)
                feat = extract_strum_frame_features(curr_lm, prev_lm)
                prev_lm = curr_lm
                frames.append(feat)
            X_list.append(frames)
            label_str = group['label'].iloc[0]
            y_list.append(STRUM_CLASSES.index(label_str) if label_str in STRUM_CLASSES else 0)
            sess_list.append(s_id)
    X_strum = np.array(X_list, dtype=np.float32)
    y_strum = np.array(y_list, dtype=np.int32)
    sess_strum = np.array(sess_list)

np.savez_compressed(
    PROCESSED_STRUM_NPZ,
    X=X_strum,
    y=y_strum,
    session_ids=sess_strum,
    class_names=np.array(STRUM_CLASSES)
)
print(f'Saved {len(X_strum)} strum sequences of shape {X_strum.shape[1:]} to {PROCESSED_STRUM_NPZ}')

Processing real raw strum frames from C:\Users\piyus\Desktop\ds cp airguitar\dataset\raw\strumming_frames.csv...
Saved 151 strum sequences of shape (15, 18) to C:\Users\piyus\Desktop\ds cp airguitar\dataset\processed\strumming_sequences.npz


## 5. Metadata and Non-Leakage Splitting Validation

In [15]:
# 5. Save metadata and verify no session leakage
meta = {
    'num_chord_samples': len(df_chords),
    'num_strum_sequences': len(X_strum),
    'chord_feature_dim': len(feature_names),
    'strum_sequence_shape': list(X_strum.shape),
    'supported_chords': SUPPORTED_CHORDS,
    'strum_classes': STRUM_CLASSES
}
with open(PROCESSED_METADATA_JSON, 'w') as f:
    json.dump(meta, f, indent=2)

# Demonstrate session-based train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(df_chords, groups=df_chords['session_id']))
tr_sess = set(df_chords.iloc[tr_idx]['session_id'].unique())
te_sess = set(df_chords.iloc[te_idx]['session_id'].unique())
overlap = tr_sess.intersection(te_sess)
assert len(overlap) == 0, 'Data leakage detected!'
print('Session-based split verified: 0 overlapping sessions between train and test.')

Session-based split verified: 0 overlapping sessions between train and test.
